In [ ]:
import torch
from torch.utils import data
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import math
import numpy as np

from datasets import Transforms, ImageNet, Places365, ArtPlaces, iNaturalist
from plot_utils import plot_points, plot_points_in_multiple_subplots
from evaluation_utils import create_model, extract_features, calculate_accuracy, calculate_precision, calculate_recall

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(torch.cuda.get_device_name())

### Konstanten

In [ ]:
from evaluation_constants import DEST, DISTANCE, K, MODELS, DATASETS

IMAGE_NUM = 1000
BATCH_SIZE = 128
FEATURE_EXTRACTION_METHOD = "t-sne"
PLOT_EACH_MODEL = False

CLASSES_IMAGENET = [
    "tabby",
    "tiger cat",
    "Persian cat",
    "Egyptian cat",
    "cougar",
    "lynx",
    "German shepherd",
    "French bulldog",
    "Eskimo dog",
    "brown bear",
    "tractor",
    "warplane",
    "passenger car"
]

CLASSES_PLACES365 = [
    "amphitheater",
    "castle",
    "palace",
    "aqueduct",
    "alley",
    "highway",
    "beer_hall",
    "chalet",
    "botanical_garden",
    "beach",
    "mountain",
    "canyon",
]

CLASSES_ARTPLACES = [
    "amphitheater",
    "castle",
    "palace",
    "aqueduct",
    "alley",
    "highway",
    "beer_hall",
    "chalet",
    "botanical_garden",
    "beach",
    "mountain",
    "canyon",
]

CLASSES_INATURALIST_FAMILY = [
    "Felidae",        # Katzen
    "Canidae",        # Hunde/Wölfe/Füchse
    "Ursidae",        # Bären
    "Accipitridae",   # Adler/Habichte
    "Anatidae",       # Enten/Gänse
    "Papilionidae",   # Schwalbenschwänze
    "Nymphalidae",    # Edelfalter
    "Formicidae",     # Ameisen
    "Apidae",         # Bienen
    "Cactaceae",      # Kakteen
    "Fagaceae",       # Eichen/Buchen
    "Pinaceae",       # Kiefern/Tannen
]

classes = {
    "ImageNet": CLASSES_IMAGENET,
    "Places365": CLASSES_PLACES365,
    "ArtPlaces": CLASSES_ARTPLACES,
    "iNaturalist_Family": CLASSES_INATURALIST_FAMILY,
}

### Datensätze

In [ ]:
imagenet = ImageNet(root=r"C:\Users\mariu\Documents\Development\Datasets\ImageNet")
imagenet_subset = imagenet.getSubset(IMAGE_NUM, CLASSES_IMAGENET)
imagenet_dataloader = data.DataLoader(imagenet_subset, batch_size=BATCH_SIZE)

places365 = Places365(root=r"C:\Users\mariu\Documents\Development\Datasets\Places365")
places365_subset = places365.getSubset(IMAGE_NUM, CLASSES_PLACES365)
places365_dataloader = data.DataLoader(places365_subset, batch_size=BATCH_SIZE)

artplaces = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280")
artplaces_subset = artplaces.getSubset(IMAGE_NUM, CLASSES_ARTPLACES)
artplaces_dataloader = data.DataLoader(artplaces_subset, batch_size=BATCH_SIZE)

inaturalist_family = iNaturalist(root=r"D:\Dokumente\Development\Datasets\iNaturalist", split="train_mini", target_type="family")
inaturalist_family_subset = inaturalist_family.getSubset(IMAGE_NUM, CLASSES_INATURALIST_FAMILY)
inaturalist_family_dataloader = data.DataLoader(inaturalist_family_subset, batch_size=BATCH_SIZE)

datasets = {
    "ImageNet": imagenet,
    "ImageNet_Loader": imagenet_dataloader,
    "Places365": places365,
    "Places365_Loader": places365_dataloader,
    "ArtPlaces": artplaces,
    "ArtPlaces_Loader": artplaces_dataloader,
    "iNaturalist_Family": inaturalist_family,
    "iNaturalist_Family_Loader": inaturalist_family_dataloader,
}

In [ ]:
def reduce_dimensions(vectors, n_components=2):
    # vectors_numpy = torch.flatten(vectors, start_dim=1).numpy()

    match FEATURE_EXTRACTION_METHOD:
        case "pca":
            fe = PCA(n_components=n_components)
            fe.fit(vectors)
            features = fe.transform(vectors)
        case "t-sne":
            fe = TSNE(n_components=n_components)
            features = fe.fit_transform(vectors)
    return features

In [ ]:
def get_features_labels_legend_colors(model, dataset, dataloader, classes, colors):
    features, labels = extract_features(model, dataloader)
    features = reduce_dimensions(features)

    legend = {}
    color_dict = {}
    for i in np.unique(labels):
        legend[i.item()] = dataset.idx_to_class[i.item()]
        if colors is not None:
            color_dict[i.item()] = colors[classes.index(legend[i.item()])]
        else:
            color_dict = None
    
    return features, labels, legend, color_dict

In [ ]:
%matplotlib qt

features_numpy_list = []
labels_numpy_list = []
legend_list = []
color_dict_list = []
title_list = []

for m in MODELS:
    print(f"Processing model: {m["name"]}")

    model = create_model(m)

    for d in DATASETS:
        features_numpy, labels_numpy, legend, color_dict = get_features_labels_legend_colors(model, datasets[d], datasets[d + "_Loader"], classes[d], None)

        # title = f"{model_info['name']} - {dataset}"
        if PLOT_EACH_MODEL:
            plot_points(features_numpy, labels_numpy, legend=legend, title=None, colors=color_dict, figsize=(15, 10), label_cols=4, fontsize=25)

        features_numpy_list.append(features_numpy)
        labels_numpy_list.append(labels_numpy)
        legend_list.append(legend)
        color_dict_list.append(color_dict)
        title_list.append(f"{m["name"]} - {datasets[d].__class__.__name__}")

    # for dataset, dataset_dataloader, classes in zip([imagenet, places365, artplaces, inaturalist_family], [imagenet_dataloader, places365_dataloader, artplaces_dataloader, inaturalist_family_dataloader], [CLASSES_IMAGENET, CLASSES_PLACES365, CLASSES_ARTPLACES, CLASSES_INATURALIST_FAMILY]):
    #     features_numpy, labels_numpy, legend, color_dict = get_features_labels_legend_colors(model, dataset, dataset_dataloader, classes, None)

    #     # title = f"{model_info['name']} - {dataset}"
    #     if PLOT_EACH_MODEL:
    #         plot_points(features_numpy, labels_numpy, legend=legend, title=None, colors=color_dict, figsize=(15, 10), label_cols=4, fontsize=25)

    #     features_numpy_list.append(features_numpy)
    #     labels_numpy_list.append(labels_numpy)
    #     legend_list.append(legend)
    #     color_dict_list.append(color_dict)
    #     title_list.append(f"{m["name"]} - {dataset.__class__.__name__}")

In [ ]:
cols = math.ceil(math.sqrt(len(features_numpy_list)))
rows = math.ceil(len(features_numpy_list) / cols)

In [ ]:
cols_without_imagenet = math.ceil(math.sqrt(len(features_numpy_list) / 2))
rows_without_imagenet = math.ceil((len(features_numpy_list) / 2) / cols)

plot_points_in_multiple_subplots(
    points=features_numpy_list[1::2],
    labels=labels_numpy_list[1::2],
    legend=legend_list[1::2],
    subplot_titles=title_list[1::2],
    nrows=rows_without_imagenet,
    ncols=cols_without_imagenet,
    colors=color_dict_list,
    label_cols=4,
    fontsize=10
)

In [ ]:
plot_points_in_multiple_subplots(
    points=features_numpy_list,
    labels=labels_numpy_list,
    legend=legend_list,
    subplot_titles=title_list,
    nrows=rows,
    ncols=cols,
    colors=color_dict_list,
    label_cols=4,
    fontsize=10
)

In [ ]:
plot_points_in_multiple_subplots(
    points=[f[0:len(f):2] for f in features_numpy_list],
    labels=[l[0:len(l):2] for l in labels_numpy_list],
    legend=legend_list,
    subplot_titles=title_list,
    nrows=rows,
    ncols=cols,
    colors=color_dict_list,
    label_cols=4,
    fontsize=10
)